# Browser Agent RL Orchestrator

**Audience:** You want a Colab-first walkthrough for browser-agent training with OpenEnv, MiniWoB, SFT, and GRPO.

**Prerequisites:** Basic Python, Colab, and high-level familiarity with LLM fine-tuning.

**Learning goals:**
- Inspect the `click-checkboxes` task interface.
- Run a baseline browser agent.
- Collect successful trajectories.
- Build an SFT dataset and prepare training.
- Prepare GRPO training from the SFT checkpoint.
- Save metrics and report artifacts cleanly.

**Outline:**
1. Validate Colab runtime and GPU.
2. Install OpenEnv, BrowserGym, and Unsloth dependencies.
3. Choose smoke/full profile and artifact root.
4. Run native probe and OpenEnv probe.
5. Run baseline evaluation and collect successful trajectories.
6. Build SFT dataset and optionally train SFT.
7. Optionally prepare and run GRPO.
8. Write final metrics/report artifacts.


## Step 1: Validate Runtime

This notebook assumes Colab with a single GPU. T4 is the primary target. The cell below fails fast if GPU is missing.

In [ ]:
import os
import platform
import torch

print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU required. Switch Colab runtime to GPU first.'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('GPU:', gpu_name)
print(f'GPU memory: {gpu_mem_gb:.1f} GB')
assert any(name in gpu_name for name in ['T4', 'L4', 'A100']), f'Unexpected GPU for this notebook: {gpu_name}'

## Step 2: Install Dependencies

These installs follow the current OpenEnv + BrowserGym + Unsloth Colab guidance. Re-run this cell if you restart the runtime.

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q "openenv-browsergym @ git+https://huggingface.co/spaces/openenv/browsergym_env"

## Step 3: Optional Google Drive Mount

Use this if you want all artifacts to land in Drive instead of ephemeral Colab disk. Set `USE_DRIVE = False` if you do not want that.

In [ ]:
from pathlib import Path

USE_DRIVE = True
ARTIFACT_ROOT = Path('/content/browser-agent-rl-artifacts')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_ROOT = Path('/content/drive/MyDrive/browser-agent-rl-artifacts')

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('Artifact root:', ARTIFACT_ROOT)

## Step 4: Load Config and Choose Run Profile

Use `smoke` first. It reduces runtime and helps verify the whole stack before longer collection or training runs.

In [ ]:
from src.config import ensure_runtime_dirs, load_config

config = load_config('configs/default.yaml')

RUN_PROFILE = 'smoke'  # change to 'full' later
if RUN_PROFILE == 'smoke':
    config['default_eval_episodes'] = 5
    config['target_successes'] = 5
    config['max_steps_per_episode'] = 6
elif RUN_PROFILE == 'full':
    pass
else:
    raise ValueError(f'Unknown RUN_PROFILE: {RUN_PROFILE}')

config['results_dir'] = str(ARTIFACT_ROOT / 'results')
config['data_dir'] = str(ARTIFACT_ROOT / 'data')
config['models_dir'] = str(ARTIFACT_ROOT / 'models')
config['sft_dataset_path'] = str(Path(config['data_dir']) / 'sft_dataset.jsonl')
config['sft_output_dir'] = str(Path(config['models_dir']) / 'sft_model')
config['grpo_output_dir'] = str(Path(config['models_dir']) / 'grpo_model')

ensure_runtime_dirs(config)
config

## Step 5: Native Probe

This is a lightweight schema sanity check. It helps confirm the task-shape assumptions before remote environment calls.

In [ ]:
from src.pipeline.run_all import run_native_probe_phase

native_probe = run_native_probe_phase()
assert 'summary' in native_probe
assert 'instruction' in native_probe['summary']
native_probe

## Step 6: OpenEnv Probe

This is the first real BrowserGym/OpenEnv call. If this fails, stop here and debug package/API/runtime issues before moving on.

In [ ]:
from src.pipeline.run_all import run_openenv_probe_phase

openenv_probe = run_openenv_probe_phase(config)
assert 'instruction' in openenv_probe
assert 'observation' in openenv_probe
print('Instruction preview:', openenv_probe['instruction'][:200])
print('Observation preview:', openenv_probe['observation'][:500])

## Step 7: Baseline Evaluation

This runs the pretrained policy through the direct OpenEnv client path and writes baseline metrics JSON.

In [ ]:
from src.pipeline.run_all import run_baseline_eval_phase

baseline_metrics = run_baseline_eval_phase(config)
baseline_metrics

## Step 8: Collect Successful Trajectories and Build SFT Dataset

This keeps only successful episodes and writes the step-wise SFT dataset to `data/sft_dataset.jsonl`.

In [ ]:
from src.pipeline.run_all import run_collection_phase, run_sft_dataset_phase

successful_episodes = run_collection_phase(config)
assert len(successful_episodes) == config['target_successes'], 'Collection did not reach target successes.'
sft_rows = run_sft_dataset_phase(config, successful_episodes)
print('Successful episodes:', len(successful_episodes))
print('SFT rows:', len(sft_rows))

## Step 9: Prepare and Optionally Run SFT

The first cell builds and prints the SFT configuration. The second cell is gated so you can inspect everything before spending GPU time.

In [ ]:
from src.train.sft import build_sft_job_config, run_sft_training

sft_config = build_sft_job_config(config['sft_dataset_path'], config['sft_output_dir'])
sft_config

In [ ]:
RUN_SFT = False

if RUN_SFT:
    sft_stats = run_sft_training(sft_config)
    print(sft_stats)
else:
    print('SFT skipped. Set RUN_SFT = True after probe/baseline/collection look correct.')

## Step 10: Prepare and Optionally Run GRPO

Only enable this after SFT has produced a usable checkpoint and OpenEnv probing is stable.

In [ ]:
from src.train.grpo import build_grpo_job_config, run_grpo_training

grpo_config = build_grpo_job_config(config['sft_output_dir'], config['grpo_output_dir'])
grpo_config

In [ ]:
RUN_GRPO = False

if RUN_GRPO:
    grpo_stats = run_grpo_training(grpo_config)
    print(grpo_stats)
else:
    print('GRPO skipped. Set RUN_GRPO = True only after SFT checkpoint exists and OpenEnv runtime is verified.')

## Step 11: Final Report Skeleton

This writes a Markdown report to the artifact root. Replace placeholder SFT/GRPO metrics after those runs finish.

In [ ]:
from src.analysis.report import write_report

metrics = {
    'base': baseline_metrics,
    'sft': {'success_rate': 0.0, 'average_reward': 0.0, 'average_steps': 0.0},
    'grpo': {'success_rate': 0.0, 'average_reward': 0.0, 'average_steps': 0.0},
}
report_text = write_report(metrics, path=str(Path(config['results_dir']) / 'report.md'))
print(report_text[:1200])

## Exercise

Try this after the smoke run works:
- switch `RUN_PROFILE` from `smoke` to `full`
- increase `default_eval_episodes`
- compare baseline metrics before enabling SFT

Answer scaffold below records what changed and what broke, if anything.

In [ ]:
notes = {
    'profile': RUN_PROFILE,
    'what_changed': '',
    'baseline_observation': '',
    'openenv_issue': '',
}
notes

## Pitfall and Extension

**Common pitfall:** OpenEnv probe fails because the remote BrowserGym environment package/API drifted. If that happens, stop after Step 6 and patch the wrapper against the live package version instead of pushing ahead.

**Optional extension:** After the one-task pipeline works, duplicate the same flow for another MiniWoB task and compare how much baseline success rate changes before any retraining.